# video-app lab 11 - Wan2.2 5B, LONGER clip (5 s target)
lab 10 measured: 832x480, 49 frames, 24 fps = 2.04 s, and the fox/forest actually looks real.
Now: 121 frames = 5.04 s at 24 fps. Memory scales with frame count, so this tries a ladder
121 -> 97 -> 81 -> 65 and keeps the longest run that fits in L4's 24 GB. Whatever wins, you get the
longest clip the hardware allows, plus the exact number for next time.
Runtime: **L4 GPU + High-RAM** (hard-stops on anything smaller).


In [ ]:
import os, shutil, psutil, torch
print(f'RAM {psutil.virtual_memory().used/1024**3:.1f}/{psutil.virtual_memory().total/1024**3:.1f} GB | disk {shutil.disk_usage("/").free/1024**3:.0f} GB')
if not torch.cuda.is_available(): raise SystemExit('STOP: no GPU -> Runtime > Change runtime type > L4 GPU + High-RAM > Save')
p = torch.cuda.get_device_properties(0); print('GPU', p.name, '| VRAM', round(p.total_memory/1024**3,1), 'GB')
if p.total_memory < 20*1024**3: raise SystemExit('STOP: <20 GB VRAM. Wan2.2 5B needs L4 (24 GB) or A100 (40 GB).')


In [ ]:
import gc, time, torch
from diffusers import AutoencoderKLWan, WanPipeline
REPO='Wan-AI/Wan2.2-TI2V-5B-Diffusers'
t0=time.time()
vae = AutoencoderKLWan.from_pretrained(REPO, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(REPO, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
try: pipe.vae.enable_tiling()
except Exception: pass
print('loaded in', round(time.time()-t0), 's | VRAM', round(torch.cuda.memory_allocated()/1024**3,2), 'GB')


In [ ]:
from diffusers.utils import export_to_video
PROMPT = ('Wide cinematic shot of a red fox trotting through deep snow between tall snow-covered pine trees, '
          'low golden winter sunbeams cutting through the branches, snow dust kicking up from its paws, '
          'fur detail, shallow depth of field, steady tracking camera, photorealistic, 35mm film')
NEG = 'blurry, low quality, distorted, extra limbs, text, watermark, flicker, jitter'
LADDER = [(121,30),(97,26),(81,22),(65,20)]
frames_out=None; used=None
for nf, st in LADDER:
    try:
        t0=time.time()
        frames_out = pipe(prompt=PROMPT, negative_prompt=NEG, height=480, width=832, num_frames=nf,
                          num_inference_steps=st, guidance_scale=5.0,
                          generator=torch.Generator('cpu').manual_seed(11)).frames[0]
        used=(nf, st); print(f'OK {nf} frames / {st} steps in {round(time.time()-t0)} s = {nf/24:.1f} s of video')
        break
    except torch.cuda.OutOfMemoryError:
        print(f'OOM at {nf} frames -> step down'); torch.cuda.empty_cache(); gc.collect()
if frames_out is None: raise SystemExit('all rungs OOM - need A100 40 GB')


In [ ]:
from IPython.display import Video, display
path=f'/content/clip_11_wan22_{used[0]}f.mp4'
export_to_video(frames_out, path, fps=24)
print('duration', round(len(frames_out)/24,2), 's | MB', round(os.path.getsize(path)/1024**2,1))
display(Video(path, embed=True, width=720))
from google.colab import files; files.download(path)


### Next after this
If 121 frames fits: 5 s clips are production-usable. Longer (10-20 s) comes from chaining -
take the last frame, feed it back as image, continue the shot. That is notebook 12.
If even 65 frames OOMs: A100 40 GB run, or sequential offload (slower but fits).
